<a href="https://colab.research.google.com/github/ngophuctaivp20-crypto/Final-Project-/blob/main/Saving_Goal_Coach_Final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

6 Steps Process:

1.understand the problem

Many people they want to save their money to buy things such as laptop, watch, or something valuable.But they don't know how to create a good saving plan. Thus, I will introduce the Saving Goal Coach that let the user set goals with deadlines, track progress,
analyse spending from a CSV of transactions, and ask an AI coach for advice based on their real numbers. It gives general budgeting guidance, not financial
advice.

2.identify the inputs and outputs

The inputs should be a savings goal, which consists of a goal name, a target amount, a deadline in YYYY-MM-DD format and any savings the user already holds. Therefore, I want the system produces output like a summary showing the amount saved, the percentage complete, the number of months remaining, the monthly contribution required to meet the deadline and an overall progress status.

Other should be transaction data, supplied as a CSV file containing the date, description, category and amount of each transaction, with income recorded as a positive value and spending as a negative value. Output would be spending totals by category, a monthly cash flow summary and a set of recommended spending reductions, together with an overall assessment of whether the user's goals are achievable.

Additionally, another input is that the user can communicate with AI and the output it will provide back is their suggesstion whether the user's budget is on tract or not and help the user what to do to achieve the goal.

3.work an example by hand,

Goal: Laptop, target 2,000, saved 500, created 26 Sep 2026, deadline 26 Mar 2027 (181 days).
- Remaining = 1500
- Months left = 181 ÷ 30 = 6 (rounded to 1 decimal)
- Needed per month = _250_
- Progress % = 25%
- Status = On track
Ask Gemini AI for the plan: Example like, "Hi, what should I do to get money in order to achieve my goal?"

4.write pseudocode

1. Settings

SET APP_CURRENCY to "AUD"
SET REQUIRED_COLUMNS to [date, description, category, amount]
SET DAYS_PER_MONTH to 30
SET CATEGORY_RULES to a dictionary: category → {cut percentage, tip}
SET DEFAULT_RULE to {cut 10%, "Review what's going into this category."}

FUNCTION load_api_key()
    TRY
        RETURN the key "GEMINI_API_KEY" from Colab Secrets
    IF that fails
        RETURN the key from the environment variables (or nothing)

SET GEMINI_API_KEY to load_api_key()
IF GEMINI_API_KEY exists
    PRINT "Gemini connected"
ELSE
    PRINT "Chat will use the offline coach"


2. Helper functions

FUNCTION savings_goal_calculator(target, saved, months)
    TRY
        turn target, saved and months into numbers
    IF that fails
        RETURN {error: "must all be numbers"}
    IF target <= 0 → RETURN {error: "Target must be more than 0"}
    IF months <= 0 → RETURN {error: "Months must be more than 0"}

    remaining ← larger of (target − saved) and 0
    per_month ← remaining ÷ months
    per_week ← per_month × 12 ÷ 52
    percent_done ← smaller of (saved ÷ target × 100) and 100
    RETURN {remaining, per_month, per_week, percent_done}

FUNCTION plan_text(plan)
    IF "error" is in plan → RETURN the error message
    IF remaining = 0      → RETURN "You have already reached your goal!"
    RETURN "You still need $X (Y% done). Save $A a month, about $B a week."

SET OPERATIONS to {"1": "How much should I save each month?", "2": "How many months will it take?"}

FUNCTION perform_calculation(operation, target, saved, value)
    IF value <= 0 → RETURN "The last number must be more than 0."
    IF operation = "1"
        RETURN plan_text(savings_goal_calculator(target, saved, value))
    ELSE IF operation = "2"
        months ← 0
        WHILE saved < target
            saved ← saved + value
            months ← months + 1
        RETURN "It will take " + months + " months."
    ELSE
        RETURN "Unknown question."

FUNCTION get_number(prompt)
    REPEAT forever
        TRY
            RETURN what the user types, turned into a number
        IF it isn't a number
            PRINT "Please type a number."

FUNCTION get_operation()
    REPEAT forever
        FOR EACH key and label in OPERATIONS
            PRINT key and label
        choice ← what the user types
        IF choice = "q" OR choice is in OPERATIONS
            RETURN choice
        PRINT "That isn't an option."

FUNCTION run_calculator()
    REPEAT forever
        operation ← get_operation()
        IF operation = "q"
            PRINT "Bye!"
            STOP the loop (break)
        target ← get_number("Target amount")
        saved ← get_number("Already saved")
        IF operation = "1"
            value ← get_number("Number of months")
        ELSE
            value ← get_number("Amount saved each month")
        PRINT perform_calculation(operation, target, saved, valu

3. Section 3: Goals (nested dictionaries)

goals ← empty dictionary          (goal name in lower case → goal dictionary)

FUNCTION total_saved(goal)
    total ← 0
    FOR EACH record in goal's contributions
        total ← total + record's amount
    RETURN total

FUNCTION percent_done(goal)
    RETURN smaller of (total_saved ÷ target × 100) and 100

FUNCTION monthly_needed(goal)
    RETURN larger of (target − total_saved) and 0, divided by months

FUNCTION status(goal)
    IF percent >= 100      → RETURN "Complete"
    ELSE IF percent >= 50  → RETURN "More than halfway"
    ELSE IF percent > 0    → RETURN "Started"
    ELSE                   → RETURN "Not started"

FUNCTION add_goal(name, target, months, saved)
    TRY
        turn target, months and saved into numbers
    IF that fails → RETURN "must all be numbers"
    IF name is empty               → RETURN "Please give your goal a name."
    IF target <= 0                 → RETURN "Target must be more than 0."
    IF months <= 0                 → RETURN "Months must be more than 0."
    IF saved < 0                   → RETURN "Already saved can't be negative."
    IF name (lower case) is in goals → RETURN "You already have that goal."
    goal ← {name, target, months, contributions: empty list}
    IF saved > 0
        ADD {saved, "Starting balance"} to the goal's contributions
    STORE goal in goals
    RETURN "Added goal. Save $X each month."

FUNCTION add_money(name, amount)
    IF name (lower case) is not in goals → RETURN "Please pick a goal."
    TRY
        turn amount into a number
    IF that fails → RETURN "Amount must be a number."
    IF amount = 0 OR total_saved + amount < 0
        RETURN "Amount can't be 0, or more than you have saved."
    ADD {amount} to the goal's contributions
    RETURN "Goal is now X% done."

FUNCTION goal_names()
    FOR EACH goal in goals → collect its name
    RETURN the list of names

FUNCTION total_monthly_needed()
    total ← 0
    FOR EACH goal in goals
        total ← total + monthly_needed(goal)
    RETURN total

FUNCTION goals_text()
    IF there are no goals → RETURN "You don't have any goals yet."
    FOR EACH goal → add a line: name, % done, status, amount per month
    RETURN the text

FUNCTION goals_table()
    FOR EACH goal → make a row
    RETURN the rows as a pandas table

4. Section 4: pandas, reading the CSV

FUNCTION load_transactions(path)
    TRY
        table ← read the CSV file
    IF that fails → RETURN False
    MAKE the column names lower case
    FOR EACH column in [date, category, amount]
        IF column is not in the table → RETURN False
    TURN the amount column into numbers (bad ones become empty)
    REMOVE rows with an empty amount
    ADD a month column, e.g. "2026-07"
    app_data["table"] ← table
    RETURN True

FUNCTION total_for(table, category)
    total ← 0
    FOR i IN range(number of rows)
        row ← row number i
        IF row's category = category AND row's amount < 0
            total ← total − amount
    RETURN total

FUNCTION spending_by_category(table)
    totals ← empty dictionary
    FOR EACH different category in the table
        spent ← total_for(table, category)
        IF spent > 0 → totals[category] ← spent
    RETURN totals

FUNCTION monthly_surplus(table)
    RETURN sum of all amounts ÷ number of months

FUNCTION suggest_cuts(table)
    surplus ← monthly_surplus(table)
    needed ← total_monthly_needed()
    text ← "Money left over each month" and "Your goals need"
    IF surplus >= needed → add "you can afford your goals"
    ELSE                 → add "you are $X short each month"
    FOR EACH category and amount spent
        IF category is not in CATEGORY_TIPS → SKIP (continue)
        saving ← spent ÷ months × the cut percentage
        IF saving < 5 → SKIP (continue)
        ADD a line with the category, the saving and the tip
    RETURN text

5. Section 5: requests, currency and the Gemini chat with custom tools

FUNCTION convert_currency(amount, from_code, to_code)
    TRY
        turn amount into a number
    IF that fails → RETURN "Amount must be a number."
    MAKE both codes upper case
    IF from_code = to_code → RETURN the same amount
    TRY
        ASK the Frankfurter website for the exchange rate
    IF that fails → RETURN "Couldn't convert."
    RETURN "amount FROM = converted TO"

SET GEMINI_TOOLS to a list of 5 tool descriptions:
    get_goals, add_money, calculate_plan, spending_tips, convert_currency

FUNCTION run_tool(name, inputs)
    IF name = "get_goals"             → RETURN goals_text()
    ELSE IF name = "add_money"        → RETURN add_money(goal name, amount)
    ELSE IF name = "calculate_plan"   → RETURN savings_goal_calculator(target, saved, months)
    ELSE IF name = "spending_tips"
        IF "table" is not in app_data → RETURN "No spending data yet."
        RETURN suggest_cuts(table)
    ELSE IF name = "convert_currency" → RETURN convert_currency(amount, from, to)
    ELSE                              → RETURN "Unknown tool"

FUNCTION ask_gemini(message)
    contents ← [the user's message]
    FOR round IN range(3)                          (at most 3 rounds)
        body ← {instructions, GEMINI_TOOLS, contents}

        FOR attempt IN range(3)                    (try again if Gemini is busy)
            TRY
                SEND body to Gemini
            IF that fails → RETURN "Gemini error: couldn't connect"
            IF the status code is not 503 → STOP trying (break)
            WAIT 3 seconds

        IF the status code is not 200 → RETURN "Gemini error" + code + reason
        TRY
            parts ← the parts of Gemini's reply
        IF that fails → RETURN "Gemini error: empty reply"
        ADD Gemini's reply to contents

        tool_answers ← empty list
        FOR EACH part in parts
            IF Gemini asked for a tool
                result ← run_tool(tool name, tool inputs)
                PRINT "Tool used: name → result"
                ADD the result to tool_answers

        IF tool_answers is empty                   (no tools needed)
            answer ← all the text parts joined together
            RETURN answer

        ADD tool_answers to contents               (send results back, go round again)
    RETURN "Sorry, that question was too complicated."

FUNCTION offline_reply(message)
    text ← message in lower case
    IF text contains "goal" OR "progress"          → RETURN run_tool("get_goals")
    ELSE IF text contains "spend" OR "save" OR "tip" → RETURN run_tool("spending_tips")
    ELSE → RETURN "Offline mode: ask about your goals or tips to save."

FUNCTION coach_reply(message, history)
    IF GEMINI_API_KEY is not ""
        answer ← ask_gemini(message)
        IF answer does not start with "Gemini error"
            RETURN answer
        PRINT the error in Colab
    RETURN offline_reply(message)                  (no key, or Gemini failed)

6. Section 6: Sample data

FUNCTION make_sample_csv()
    one_month ← list of 7 transactions (description, category, amount)
    rows ← empty list
    FOR EACH month in ["2026-07", "2026-08", "2026-09"]
        FOR EACH transaction in one_month
            ADD {date, description, category, amount} to rows
    SAVE rows as "sample_transactions.csv"
    RETURN the file name

7. Section 7: Tests

FUNCTION run_tests()
    CHECK money() formats numbers correctly
    CHECK savings_goal_calculator gives the right plan, and errors for bad input
    CHECK perform_calculation answers both questions
    CHECK add_goal and add_money work, and reject bad input
    WRITE a small test CSV file
    CHECK load_transactions removes bad rows, and returns False for a missing file
    CHECK total_for, spending_by_category, monthly_surplus and suggest_cuts
    CHECK convert_currency (the parts that don't need the internet)
    CHECK every custom tool in run_tool, including an unknown tool name
    CLEAR goals and app_data
    PRINT "All tests passed"

8. Section 8: Gradio app

FUNCTION ui_add_goal(name, target, months, saved)
    RETURN add_goal message, goals table, updated goal dropdown

FUNCTION ui_add_money(name, amount)
    RETURN add_money message, goals table

FUNCTION ui_load(path)
    IF load_transactions(path) = False → RETURN "Couldn't read that file"
    RETURN "Loaded X transactions", spending table, suggest_cuts text

FUNCTION ui_sample()
    RETURN ui_load(make_sample_csv())

FUNCTION ui_calculator(target, saved, months)
    RETURN plan_text(savings_goal_calculator(target, saved, months))

BUILD the window with 5 tabs:
    Goals       → add a goal, add money, goals table
    Calculator  → goal amount, saved, months, Calculate button
    Spending    → upload CSV, sample data button, spending table, suggestions
    Currency    → amount, from, to, Convert button
    Coach       → chat box that uses coach_reply
CONNECT each button to its function
LAUNCH the app

  

5.convert pseudocode to Python

6.test with a variety of data.